In [ ]:
# Select Planet and Folder Structure
from typing import Literal
from pathlib import Path

import numpy as np
import ipywidgets
import matplotlib.pyplot as plt

from PIL import Image
from numpy.typing import NDArray

from mirage_texture_parser import MirageTextureLoader


def list_dir(folder: str | Path) -> list[Path]:
    """Scan a folder and return all folders inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    return [f for f in folder.iterdir() if f.is_dir()]


def list_files(folder: str | Path, extension: str | None = None) -> list[Path]:
    """Scan a folder and return all files inside."""
    folder = folder if isinstance(folder, Path) else Path(folder)
    output_files = []
    for f in folder.iterdir():
        if f.is_file():
            if extension is not None:
                if f.suffix != extension:
                    continue
            output_files.append(f)
    return output_files


def detect_cubemap_bodies(folder: Path) -> dict[str, Path]:
    """Find body information for values within the folder."""
    bodies = {}

    def iter_search(fpath: Path):
        dirs = list_dir(fpath)
        if "Terrain" in [d.stem for d in dirs]:
            bodies[fpath.stem.rsplit("_", 1)[1]] = fpath / Path("Terrain")
            return
        for d in dirs:
            iter_search(d)

    iter_search(folder)
    return bodies


sol_dir = r"C:\Users\rweld\Documents\Kerbal Space Program 1\KSP Mirage Beta\GameData\Sol-Textures\PluginData"

bodies = {}
for f in list_dir(sol_dir):
    if f.name[0].isdigit():
        if int(f.name.split("_", 1)[0]):
            bodies.update(detect_cubemap_bodies(f))

select_widget = ipywidgets.Select(
    options=list(bodies.keys()),
    description="Select Body:",
    disabled=False,
)

select_widget

In [ ]:
# Define Cube Map Allocations
body_name = select_widget.value


class CubeMapPlanet:
    """Pull the files from Terrain to make a LOD cube map!"""

    def __init__(self, fpath: str | Path):
        self.folder = fpath if isinstance(fpath, Path) else Path(fpath)
        self.num_layers = len(list_dir(self.folder))

    def get_image(self, face: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"]):
        """Pull image."""

    def stitch_image(
        self, face: Literal["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"], lod: int = 0
    ):
        """Get the combined image from the tiles in the texture."""
        assert lod < self.num_layers
        images = []

        textures = MirageTextureLoader(self.folder, lod)
        assert textures.colour_idx.num_entries == 6 * 4**lod  # 1, 4, 16 tex per face
        tex_per_face = textures.colour_idx.num_entries // 6
        tex_order = ["Xp", "Xn", "Yn", "Yp", "Zn", "Zp"]

        if lod == 0:
            return textures.get_texture(tex_order.index(face))

        order = [[0, 2], [1, 3]]
        indices = order
        for _ in range(lod - 1):
            size = len(indices) ** 2
            first_half = indices + [[x + size for x in segment] for segment in indices]
            last_half = [[x + 2 * size for x in segment] for segment in first_half]
            indices = [x[0] + x[1] for x in zip(first_half, last_half)]

        indices = [x for xs in indices for x in xs]

        for i in indices:
            im_array = textures.get_texture(tex_order.index(face) * tex_per_face + i)
            images.append(im_array)

        lines = []
        tile_count = 2**lod
        for i in range(tile_count):
            # Stitch while keeping padding in place
            imgs = images[tile_count * i : tile_count * (i + 1)]
            for j in range(tile_count):
                start = 0 if j == 0 else 4
                end = 264 if j == tile_count - 1 else 260
                imgs[j] = imgs[j][start:end, :, :]

            start = 0 if i == 0 else 4
            end = 264 if i == tile_count - 1 else 260
            imgs = [x[:, start:end, :] for x in imgs]  # Remove padding on middle layers
            res = np.concatenate(imgs, axis=0)
            lines.append(res)

        return np.concatenate(lines, axis=1)


dd = CubeMapPlanet(bodies[body_name])
data = dd.stitch_image("Yn", 3)

# Display Image
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.axis("off")
Image.fromarray(data).save("image.png")
plt.imshow(data)
plt.show()

In [ ]:
# Work out cube to rectangle projection

# u = 0 to 2 pi
# v = -pi / 2 to pi / 2


def bilinear_interpolate(arr: NDArray, x: float, y: float) -> tuple[int, int, int, int]:
    """
    Interpolate a value at (x, y) in a 2D numpy array arr.
    """
    # 1. Identify surrounding grid indices
    x1, y1 = int(np.floor(x)), int(np.floor(y))
    x2, y2 = x1 + 1, y1 + 1

    # Boundary checks to prevent index errors
    x2 = min(x2, arr.shape[1] - 1)
    y2 = min(y2, arr.shape[0] - 1)

    # 2. Extract the four nearest neighbor values
    p11 = tuple(arr[y1, x1, :])  # Bottom-left
    p12 = tuple(arr[y2, x1, :])  # Top-left
    p21 = tuple(arr[y1, x2, :])  # Bottom-right
    p22 = tuple(arr[y2, x2, :])  # Top-right

    # 3. Calculate relative distances (weights)
    x_diff = x - x1
    y_diff = y - y1

    # 4. Compute weighted average
    # Formula: (1-dx)(1-dy)*p11 + (1-dx)*dy*p21 + dx*(1-dy)*p12 + dx*dy*p22
    res = []
    for channel in range(len(p11)):
        interpolated = (
            (1 - x_diff) * (1 - y_diff) * p11[channel]
            + (1 - x_diff) * y_diff * p21[channel]
            + x_diff * (1 - y_diff) * p12[channel]
            + x_diff * y_diff * p22[channel]
        )
        res.append(interpolated)

    return tuple(res)


def project_uv(cube: CubeMapPlanet, width: int, height: int) -> NDArray:
    """Cube Map to Equirectangular Projection."""

    w_scale = 2 * np.pi / width
    h_scale = np.pi / height

    cube_sides: dict[str, NDArray] = {}
    for side in ["Xn", "Xp", "Yn", "Yp", "Zn", "Zp"]:
        cube_sides[side] = cube.stitch_image(side, 1)  # type: ignore[reportTypeArgument]

    pixel = np.zeros([height, width, 4], dtype=np.uint8)
    padding = 4  # Number of padded pixels around each image

    for h_pixel in range(height):
        for w_pixel in range(width):
            u = w_pixel * w_scale - 3 * np.pi / 8
            v = h_pixel * h_scale - np.pi / 2

            x = np.sin(u) * np.cos(v)
            y = np.sin(v)
            z = np.cos(u) * np.cos(v)

            a = max(np.abs(x), np.abs(y), np.abs(z))

            if a == np.abs(x):
                side = "Xp" if x > 0 else "Xn"
                s = cube_sides[side].shape[0] - 2 * padding - 1
                p1 = s * (x + y * np.sign(x)) / (2 * x)
                p2 = s * (x + z) / (2 * x)
            elif a == np.abs(y):
                side = "Yp" if y > 0 else "Yn"
                s = cube_sides[side].shape[0] - 2 * padding - 1
                p1 = s * (y + z) / (2 * y)
                p2 = s * (y + x * np.sign(y)) / (2 * y)
            else:
                side = "Zp" if z > 0 else "Zn"
                s = cube_sides[side].shape[0] - 2 * padding - 1
                p1 = s * (z + y * np.sign(z)) / (2 * z)
                p2 = s * (z - x) / (2 * z)

            # p = bilinear_interpolate(cube_sides[side], p2 + padding, p1 + padding)
            # pixel[h_pixel, w_pixel, :] = p
            p1 = int(round(p1))
            p2 = int(round(p2))
            pixel[h_pixel, w_pixel, :] = cube_sides[side][p1 + padding, p2 + padding, :]

    return pixel


proj_data = project_uv(dd, 2048, 1024)
print(f"End data shape {proj_data.shape}")

# Display Image
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
plt.axis("off")
Image.fromarray(proj_data).save("image.png")
plt.imshow(proj_data)
plt.show()